In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, InceptionV3, DenseNet121
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path
import warnings
import gc
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
import time
import zipfile
import io
warnings.filterwarnings('ignore')

# Custom layer for channel splitting (replaces Lambda layers for safe serialization)
@keras.utils.register_keras_serializable(package="Custom")
class SplitChannels(layers.Layer):
    """Custom layer to split channels from input tensor."""
    def __init__(self, start_channel, end_channel, **kwargs):
        super(SplitChannels, self).__init__(**kwargs)
        self.start_channel = start_channel
        self.end_channel = end_channel
    
    def call(self, inputs):
        return inputs[:, :, :, self.start_channel:self.end_channel]
    
    def get_config(self):
        config = super(SplitChannels, self).get_config()
        config.update({
            'start_channel': self.start_channel,
            'end_channel': self.end_channel
        })
        return config

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 50

# Simplified model names for plots
SIMPLE_NAMES = {
    'DenseNet121_v1_original': 'DN121-Frozen',
    'DenseNet121_v2_finetune': 'DN121-Tuned',
    'MobileNetV2_v1_original': 'MobileV2-Frozen',
    'MobileNetV2_v2_finetune': 'MobileV2-Tuned',    
    'InceptionV3_v1_original': 'InceptV3-Frozen',
    'InceptionV3_v2_finetune': 'InceptV3-Tuned',
    'Ensemble 1: DenseNet121 v1+v2': 'Ens-DN-Frozen+Tuned',
    'Ensemble 2: DenseNet121 v2 + MobileNetV2 v2': 'Ens-DN+Mob-Tuned',
    'Ensemble 3: DenseNet121 v2 + InceptionV3 v2': 'Ens-DN+Incept-Tuned'
}

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# ==================== 5 Model Configurations ====================
MODEL_VERSIONS = {
    'DenseNet121_v1_original': {
        'base_model': DenseNet121,
        'display_name': 'DenseNet121 v1 (Original)',
        'learning_rate': 0.001,
        'fine_tune': False,
        'fine_tune_at': 0,
        'dropout': [0.5, 0.3, 0.2],
    },
    'DenseNet121_v2_finetune': {
        'base_model': DenseNet121,
        'display_name': 'DenseNet121 v2 (Fine-tuned)',
        'learning_rate': 0.0005,
        'fine_tune': True,
        'fine_tune_at': 100,
        'dropout': [0.5, 0.3, 0.2],
    },
    'MobileNetV2_v1_original': {
        'base_model': MobileNetV2,
        'display_name': 'MobileNetV2 v1 (Original)',
        'learning_rate': 0.001,
        'fine_tune': False,
        'fine_tune_at': 0,
        'dropout': [0.5, 0.3, 0.2],
    },
    'MobileNetV2_v2_finetune': {
        'base_model': MobileNetV2,
        'display_name': 'MobileNetV2 v2 (Fine-tuned)',
        'learning_rate': 0.0005,
        'fine_tune': True,
        'fine_tune_at': 80,
        'dropout': [0.5, 0.3, 0.2],
    },
    'InceptionV3_v1_original': {
        'base_model': InceptionV3,
        'display_name': 'InceptionV3 v1 (Original)',
        'learning_rate': 0.001,
        'fine_tune': False,
        'fine_tune_at': 0,
        'dropout': [0.5, 0.3, 0.2],
    },
    'InceptionV3_v2_finetune': {
        'base_model': InceptionV3,
        'display_name': 'InceptionV3 v2 (Fine-tuned)',
        'learning_rate': 0.0005,
        'fine_tune': True,
        'fine_tune_at': 200,
        'dropout': [0.5, 0.3, 0.2],
    },
}

# Paths
METADATA_PATH = '/kaggle/input/cattle4sides/cattleMetadata.csv'
IMAGES_BASE_PATH = '/kaggle/input/cattle4sides/cattle4sides/4 Sides'

# ==================== Memory Management ====================

def get_memory_usage():
    """Get current RAM and GPU memory usage."""
    process = psutil.Process()
    ram_mb = process.memory_info().rss / 1024 / 1024
    system_ram = psutil.virtual_memory()
    ram_percent = system_ram.percent
    
    gpu_memory = {}
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            gpu_info = tf.config.experimental.get_memory_info('GPU:0')
            gpu_current_mb = gpu_info['current'] / 1024 / 1024
            gpu_peak_mb = gpu_info['peak'] / 1024 / 1024
            gpu_memory = {
                'current_mb': gpu_current_mb,
                'peak_mb': gpu_peak_mb
            }
        except:
            gpu_memory = {'current_mb': 0, 'peak_mb': 0}
    
    return {
        'ram_mb': ram_mb,
        'ram_percent': ram_percent,
        'gpu_memory': gpu_memory
    }

def clean_memory():
    """Clean both system RAM and GPU memory."""
    print("\n" + "="*60)
    print("MEMORY CLEANUP")
    print("="*60)
    
    before = get_memory_usage()
    print(f"\nBefore Cleanup:")
    print(f"  RAM: {before['ram_mb']:.2f} MB ({before['ram_percent']:.1f}% system usage)")
    if before['gpu_memory']:
        print(f"  GPU: Current={before['gpu_memory']['current_mb']:.2f} MB, Peak={before['gpu_memory']['peak_mb']:.2f} MB")
    
    keras.backend.clear_session()
    gc.collect()
    
    if tf.config.list_physical_devices('GPU'):
        try:
            tf.keras.backend.clear_session()
            tf.compat.v1.reset_default_graph()
        except:
            pass
    
    after = get_memory_usage()
    print(f"\nAfter Cleanup:")
    print(f"  RAM: {after['ram_mb']:.2f} MB ({after['ram_percent']:.1f}% system usage)")
    if after['gpu_memory']:
        print(f"  GPU: Current={after['gpu_memory']['current_mb']:.2f} MB, Peak={after['gpu_memory']['peak_mb']:.2f} MB")
    
    ram_freed = before['ram_mb'] - after['ram_mb']
    print(f"\nRAM Freed: {ram_freed:.2f} MB")
    if before['gpu_memory'] and after['gpu_memory']:
        gpu_freed = before['gpu_memory']['current_mb'] - after['gpu_memory']['current_mb']
        print(f"GPU Freed: {gpu_freed:.2f} MB")
    print("="*60 + "\n")

# ==================== Data Loading ====================

def find_image_flexible(base_path, cow_id, side):
    """
    Flexibly find images with different naming conventions.
    Handles: serialNo_side.jpg, side.jpg, serialNo_side.JPG, side.JPG, etc.
    """
    if pd.isna(cow_id):
        return None
    
    cow_id_str = str(int(cow_id)) if isinstance(cow_id, float) else str(cow_id)
    cow_dir = os.path.join(base_path, cow_id_str)
    
    if not os.path.exists(cow_dir):
        return None
    
    all_files = os.listdir(cow_dir)
    
    # Normalize side name
    side_lower = side.lower()
    
    # Try to find matching file
    for filename in all_files:
        filename_lower = filename.lower()
        
        # Check if the side is in the filename
        if side_lower in filename_lower:
            # Verify it's an image file
            if filename_lower.endswith(('.jpg', '.jpeg', '.png')):
                return os.path.join(cow_dir, filename)
    
    return None

def load_and_preprocess_image(image_path, target_size=(IMG_SIZE, IMG_SIZE)):
    """Load and preprocess a single image."""
    try:
        img = load_img(image_path, target_size=target_size)
        img_array = img_to_array(img)
        img_array = img_array / 255.0
        return img_array
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return None

def load_dataset(metadata_path, images_base_path):
    """Load and preprocess the cattle dataset with 4-view images."""
    print(f"Loading metadata from: {metadata_path}")
    print(f"Loading images from: {images_base_path}")
    
    df = pd.read_csv(metadata_path)
    
    weight_col = 'Weight(KG)' if 'Weight(KG)' in df.columns else 'Weight'
    id_col = 'ID' if 'ID' in df.columns else 'id'
    
    print(f"\nDataset Info:")
    print(f"Total records: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Weight range: {df[weight_col].min():.2f} - {df[weight_col].max():.2f} kg")
    
    X_combined = []
    y = []
    valid_ids = []
    valid_indices = []
    sides = ['front', 'back', 'left', 'right']
    
    print("\nLoading images...")
    for idx, row in df.iterrows():
        cow_id = row[id_col]
        weight = row[weight_col]
        
        # Skip rows with NaN ID or weight
        if pd.isna(cow_id) or pd.isna(weight):
            print(f"Skipping row {idx} - NaN ID or weight")
            continue
        
        # Load all 4 sides
        images = []
        all_found = True
        
        for side in sides:
            img_path = find_image_flexible(images_base_path, cow_id, side)
            if img_path and os.path.exists(img_path):
                img = load_and_preprocess_image(img_path)
                if img is not None:
                    images.append(img)
                else:
                    all_found = False
                    break
            else:
                all_found = False
                break
        
        if all_found and len(images) == 4:
            combined_image = np.concatenate(images, axis=-1)
            X_combined.append(combined_image)
            y.append(weight)
            valid_ids.append(cow_id)
            valid_indices.append(idx)
        else:
            print(f"Missing images for cow ID {cow_id}")
        
        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{len(df)} records...")
    
    X = np.array(X_combined)
    y = np.array(y)
    
    print(f"\nDataset loaded successfully!")
    print(f"Valid samples: {len(X)}")
    print(f"Input shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    print(f"Weight range: {y.min():.2f} - {y.max():.2f} kg")
    
    return X, y, valid_ids

# ==================== Model Building ====================

def build_model(base_model_func, input_shape, model_name, config):
    """Build a regression model with specified configuration."""
    print(f"\nBuilding model: {model_name}")
    print(f"  Config: LR={config['learning_rate']}, Fine-tune={config['fine_tune']}")
    
    image_input = layers.Input(shape=input_shape, name='image_input')
    
    # Split the 12 channels into 4 separate RGB images using custom SplitChannels layer
    front = SplitChannels(0, 3, name='split_front')(image_input)
    back = SplitChannels(3, 6, name='split_back')(image_input)
    left = SplitChannels(6, 9, name='split_left')(image_input)
    right = SplitChannels(9, 12, name='split_right')(image_input)
    
    # Create base model (pretrained on ImageNet)
    base_model = base_model_func(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    # Fine-tuning configuration
    if config['fine_tune']:
        base_model.trainable = True
        fine_tune_at = config['fine_tune_at']
        for layer in base_model.layers[:fine_tune_at]:
            layer.trainable = False
        print(f"  Fine-tuning: Unfrozen {len(base_model.layers) - fine_tune_at} layers (from layer {fine_tune_at})")
    else:
        base_model.trainable = False
        print(f"  All base layers frozen")
    
    # Process each view separately
    front_features = base_model(front)
    back_features = base_model(back)
    left_features = base_model(left)
    right_features = base_model(right)
    
    # Global average pooling for each view
    front_pool = layers.GlobalAveragePooling2D()(front_features)
    back_pool = layers.GlobalAveragePooling2D()(back_features)
    left_pool = layers.GlobalAveragePooling2D()(left_features)
    right_pool = layers.GlobalAveragePooling2D()(right_features)
    
    # Concatenate all image features
    image_combined = layers.Concatenate(name='image_features')([front_pool, back_pool, left_pool, right_pool])
    
    # Regression head
    dropout_vals = config['dropout']
    x = layers.Dense(512, activation='relu')(image_combined)
    x = layers.Dropout(dropout_vals[0])(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(dropout_vals[1])(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(dropout_vals[2])(x)
    outputs = layers.Dense(1, activation='linear', name='weight_output')(x)
    
    model = keras.Model(inputs=image_input, outputs=outputs, name=model_name)
    
    return model

# ==================== Training ====================

def train_and_evaluate(model, X_train, y_train, X_val, y_val, model_name, learning_rate):
    """Train and evaluate a model."""
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae', 'mse']
    )
    
    # Callbacks
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=8,
            min_lr=1e-7,
            verbose=1
        )
    ]
    
    # Train with timing
    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )
    training_time = time.time() - start_time
    
    # Evaluate with timing
    start_time = time.time()
    y_pred_train = model.predict(X_train, verbose=0).flatten()
    train_inference_time = time.time() - start_time
    
    start_time = time.time()
    y_pred_val = model.predict(X_val, verbose=0).flatten()
    val_inference_time = time.time() - start_time
    
    train_mae = mean_absolute_error(y_train, y_pred_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    train_r2 = r2_score(y_train, y_pred_train)
    
    val_mae = mean_absolute_error(y_val, y_pred_val)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    val_r2 = r2_score(y_val, y_pred_val)
    
    print(f"\n{model_name} Results:")
    print(f"Train - MAE: {train_mae:.2f}, RMSE: {train_rmse:.2f}, R²: {train_r2:.4f}")
    print(f"Val   - MAE: {val_mae:.2f}, RMSE: {val_rmse:.2f}, R²: {val_r2:.4f}")
    print(f"Training Time: {training_time:.2f}s, Inference Time: {val_inference_time:.4f}s")
    
    results = {
        'model_name': model_name,
        'display_name': MODEL_VERSIONS[model_name]['display_name'],
        'train_mae': train_mae,
        'train_rmse': train_rmse,
        'train_r2': train_r2,
        'val_mae': val_mae,
        'val_rmse': val_rmse,
        'val_r2': val_r2,
        'epochs_trained': len(history.history['loss']),
        'training_time': training_time,
        'train_inference_time': train_inference_time,
        'val_inference_time': val_inference_time,
        'history': history.history
    }
    
    return results, model

# ==================== Plotting Functions ====================

def plot_predicted_vs_actual(y_true, y_pred, model_name, dataset_type):
    """Plot predicted vs actual weights with regression line. Returns BytesIO and filename."""
    plt.figure(figsize=(10, 8))
    
    # Scatter plot
    plt.scatter(y_true, y_pred, alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
    
    # Perfect prediction line
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    
    # Calculate metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    plt.xlabel('Actual Weight (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Predicted Weight (kg)', fontsize=14, fontweight='bold')
    plt.title(f'{model_name} - Predicted vs Actual ({dataset_type})\nMAE: {mae:.2f} kg, RMSE: {rmse:.2f} kg, R²: {r2:.4f}', 
              fontsize=14, fontweight='bold', pad=20)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_pred_vs_actual_{dataset_type.lower()}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_residuals(y_true, y_pred, model_name, dataset_type):
    """Plot residual plot to check for patterns. Returns BytesIO and filename."""
    residuals = y_true - y_pred
    
    plt.figure(figsize=(10, 8))
    plt.scatter(y_pred, residuals, alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
    plt.axhline(y=0, color='r', linestyle='--', lw=2, label='Zero Error')
    
    plt.xlabel('Predicted Weight (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Residuals (kg)', fontsize=14, fontweight='bold')
    plt.title(f'{model_name} - Residual Plot ({dataset_type})', fontsize=14, fontweight='bold', pad=20)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_residuals_{dataset_type.lower()}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_error_distribution(y_true, y_pred, model_name, dataset_type):
    """Plot distribution of prediction errors. Returns BytesIO and filename."""
    errors = y_true - y_pred
    
    plt.figure(figsize=(10, 8))
    plt.hist(errors, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
    plt.axvline(x=0, color='r', linestyle='--', lw=2, label='Zero Error')
    plt.axvline(x=errors.mean(), color='green', linestyle='--', lw=2, label=f'Mean Error: {errors.mean():.2f} kg')
    
    plt.xlabel('Error (Actual - Predicted) (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Frequency', fontsize=14, fontweight='bold')
    plt.title(f'{model_name} - Error Distribution ({dataset_type})\nStd: {errors.std():.2f} kg', 
              fontsize=14, fontweight='bold', pad=20)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_error_dist_{dataset_type.lower()}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_absolute_errors(y_true, y_pred, model_name, dataset_type):
    """Plot absolute errors vs actual weights. Returns BytesIO and filename."""
    abs_errors = np.abs(y_true - y_pred)
    
    plt.figure(figsize=(10, 8))
    plt.scatter(y_true, abs_errors, alpha=0.6, s=50, edgecolors='k', linewidth=0.5, c=abs_errors, cmap='YlOrRd')
    plt.colorbar(label='Absolute Error (kg)')
    
    mae = mean_absolute_error(y_true, y_pred)
    plt.axhline(y=mae, color='blue', linestyle='--', lw=2, label=f'MAE: {mae:.2f} kg')
    
    plt.xlabel('Actual Weight (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Absolute Error (kg)', fontsize=14, fontweight='bold')
    plt.title(f'{model_name} - Absolute Errors ({dataset_type})', fontsize=14, fontweight='bold', pad=20)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_abs_errors_{dataset_type.lower()}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_metrics_comparison(results_df, metric_name, ylabel, title, filename):
    """Plot comparison of a specific metric across all models. Returns BytesIO and filename."""
    plt.figure(figsize=(14, 8))
    
    x_pos = np.arange(len(results_df))
    colors = ['#1f77b4' if 'Ens' not in name else '#ff7f0e' for name in results_df['simple_name']]
    
    bars = plt.bar(x_pos, results_df[metric_name], color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, results_df[metric_name])):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.xlabel('Model', fontsize=14, fontweight='bold')
    plt.ylabel(ylabel, fontsize=14, fontweight='bold')
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.xticks(x_pos, results_df['simple_name'], rotation=45, ha='right', fontsize=11)
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#1f77b4', edgecolor='black', label='Individual Models'),
                      Patch(facecolor='#ff7f0e', edgecolor='black', label='Ensemble Models')]
    plt.legend(handles=legend_elements, fontsize=12, loc='upper right')
    
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_train_val_comparison(results_df, metric_train, metric_val, ylabel, title, filename):
    """Plot training vs validation metrics comparison. Returns BytesIO and filename."""
    plt.figure(figsize=(14, 8))
    
    x_pos = np.arange(len(results_df))
    width = 0.35
    
    bars1 = plt.bar(x_pos - width/2, results_df[metric_train], width, label='Training', 
                    color='#2ecc71', edgecolor='black', linewidth=1.5, alpha=0.8)
    bars2 = plt.bar(x_pos + width/2, results_df[metric_val], width, label='Validation', 
                    color='#e74c3c', edgecolor='black', linewidth=1.5, alpha=0.8)
    
    plt.xlabel('Model', fontsize=14, fontweight='bold')
    plt.ylabel(ylabel, fontsize=14, fontweight='bold')
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.xticks(x_pos, results_df['simple_name'], rotation=45, ha='right', fontsize=11)
    plt.legend(fontsize=12, loc='upper right')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_all_metrics_heatmap(results_df):
    """Plot heatmap of all metrics for all models. Returns BytesIO and filename."""
    plt.figure(figsize=(12, 10))
    
    metrics_data = results_df[['train_mae', 'train_rmse', 'train_r2', 'val_mae', 'val_rmse', 'val_r2']].values
    
    sns.heatmap(metrics_data, annot=True, fmt='.3f', cmap='RdYlGn_r', 
                xticklabels=['Train MAE', 'Train RMSE', 'Train R²', 'Val MAE', 'Val RMSE', 'Val R²'],
                yticklabels=results_df['simple_name'], cbar_kws={'label': 'Metric Value'},
                linewidths=0.5, linecolor='black')
    
    plt.title('Performance Metrics Heatmap - All Models', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Metrics', fontsize=14, fontweight='bold')
    plt.ylabel('Model', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)
    plt.tight_layout()
    
    filename = 'all_metrics_heatmap.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_error_boxplot(all_model_errors):
    """Plot boxplot of errors for all models. Returns BytesIO and filename."""
    plt.figure(figsize=(14, 8))
    
    bp = plt.boxplot([errors for _, errors in all_model_errors], 
                     labels=[name for name, _ in all_model_errors],
                     patch_artist=True, showmeans=True,
                     meanprops=dict(marker='D', markerfacecolor='red', markersize=8))
    
    # Color the boxes
    colors = ['lightblue' if 'Ens' not in name else 'lightcoral' for name, _ in all_model_errors]
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    plt.xlabel('Model', fontsize=14, fontweight='bold')
    plt.ylabel('Prediction Error (kg)', fontsize=14, fontweight='bold')
    plt.title('Prediction Error Distribution Across All Models', fontsize=16, fontweight='bold', pad=20)
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.axhline(y=0, color='green', linestyle='--', lw=2, alpha=0.5, label='Zero Error')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    filename = 'error_boxplot_all_models.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_ensemble_improvement(results_df):
    """Plot ensemble improvement over base models. Returns BytesIO and filename."""
    # Extract base models and ensembles
    base_models = results_df[~results_df['simple_name'].str.contains('Ens')]
    ensembles = results_df[results_df['simple_name'].str.contains('Ens')]
    
    if len(ensembles) == 0:
        return None, None
    
    plt.figure(figsize=(12, 8))
    
    # Calculate average of base models
    base_avg_mae = base_models['val_mae'].mean()
    
    x_pos = np.arange(len(ensembles))
    improvements = [(base_avg_mae - mae) / base_avg_mae * 100 for mae in ensembles['val_mae']]
    
    colors = ['green' if imp > 0 else 'red' for imp in improvements]
    bars = plt.bar(x_pos, improvements, color=colors, edgecolor='black', linewidth=1.5, alpha=0.7)
    
    # Add value labels
    for bar, imp in zip(bars, improvements):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{imp:+.2f}%', ha='center', va='bottom' if imp > 0 else 'top', 
                fontsize=11, fontweight='bold')
    
    plt.axhline(y=0, color='black', linestyle='-', lw=1)
    plt.xlabel('Ensemble Model', fontsize=14, fontweight='bold')
    plt.ylabel('Improvement over Base Models Average (%)', fontsize=14, fontweight='bold')
    plt.title(f'Ensemble Performance Improvement\n(Base Models Avg MAE: {base_avg_mae:.2f} kg)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xticks(x_pos, ensembles['simple_name'], rotation=45, ha='right', fontsize=11)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    filename = 'ensemble_improvement.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_model_ranking(results_df):
    """Plot model ranking based on validation MAE. Returns BytesIO and filename."""
    plt.figure(figsize=(12, 8))
    
    sorted_df = results_df.sort_values('val_mae')
    ranks = np.arange(1, len(sorted_df) + 1)
    
    colors = ['gold' if i == 0 else 'silver' if i == 1 else 'chocolate' if i == 2 else 'steelblue' 
              for i in range(len(sorted_df))]
    
    bars = plt.barh(ranks, sorted_df['val_mae'], color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)
    
    # Add value labels
    for bar, val in zip(bars, sorted_df['val_mae']):
        width = bar.get_width()
        plt.text(width, bar.get_y() + bar.get_height()/2.,
                f' {val:.2f} kg', ha='left', va='center', fontsize=10, fontweight='bold')
    
    plt.ylabel('Rank', fontsize=14, fontweight='bold')
    plt.xlabel('Validation MAE (kg)', fontsize=14, fontweight='bold')
    plt.title('Model Ranking by Validation MAE (Lower is Better)', fontsize=16, fontweight='bold', pad=20)
    plt.yticks(ranks, [f'#{r}: {name}' for r, name in zip(ranks, sorted_df['simple_name'])], fontsize=11)
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    
    filename = 'model_ranking.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_r2_comparison_scatter(results_df):
    """Plot R² scores for train vs validation. Returns BytesIO and filename."""
    plt.figure(figsize=(10, 10))
    
    colors = ['red' if 'Ens' in name else 'blue' for name in results_df['simple_name']]
    
    plt.scatter(results_df['train_r2'], results_df['val_r2'], s=200, c=colors, 
               alpha=0.6, edgecolors='black', linewidth=2)
    
    # Add perfect correlation line
    min_r2 = min(results_df['train_r2'].min(), results_df['val_r2'].min())
    max_r2 = max(results_df['train_r2'].max(), results_df['val_r2'].max())
    plt.plot([min_r2, max_r2], [min_r2, max_r2], 'k--', lw=2, label='Perfect Agreement')
    
    # Add labels for each point
    for idx, row in results_df.iterrows():
        plt.annotate(row['simple_name'], (row['train_r2'], row['val_r2']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')
    
    plt.xlabel('Training R²', fontsize=14, fontweight='bold')
    plt.ylabel('Validation R²', fontsize=14, fontweight='bold')
    plt.title('R² Score: Training vs Validation', fontsize=16, fontweight='bold', pad=20)
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='blue', edgecolor='black', label='Individual Models', alpha=0.6),
                      Patch(facecolor='red', edgecolor='black', label='Ensemble Models', alpha=0.6)]
    plt.legend(handles=legend_elements, fontsize=12, loc='lower right')
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    filename = 'r2_train_vs_val_scatter.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_overfitting_analysis(results_df):
    """Plot overfitting analysis (train-val gap). Returns BytesIO and filename."""
    plt.figure(figsize=(14, 8))
    
    mae_gap = results_df['train_mae'] - results_df['val_mae']
    rmse_gap = results_df['train_rmse'] - results_df['val_rmse']
    
    x_pos = np.arange(len(results_df))
    width = 0.35
    
    bars1 = plt.bar(x_pos - width/2, mae_gap, width, label='MAE Gap', 
                    color='coral', edgecolor='black', linewidth=1.5, alpha=0.8)
    bars2 = plt.bar(x_pos + width/2, rmse_gap, width, label='RMSE Gap', 
                    color='skyblue', edgecolor='black', linewidth=1.5, alpha=0.8)
    
    plt.axhline(y=0, color='red', linestyle='--', lw=2, label='No Gap')
    plt.xlabel('Model', fontsize=14, fontweight='bold')
    plt.ylabel('Training - Validation Gap (kg)', fontsize=14, fontweight='bold')
    plt.title('Overfitting Analysis: Train-Val Performance Gap\n(Negative = Better on Validation)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xticks(x_pos, results_df['simple_name'], rotation=45, ha='right', fontsize=11)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    filename = 'overfitting_analysis.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_architecture_comparison(results_df):
    """Compare performance by architecture type. Returns BytesIO and filename."""
    plt.figure(figsize=(12, 8))
    
    # Group by architecture
    arch_groups = {
        'DenseNet121': results_df[results_df['simple_name'].str.contains('DN121')],
        'MobileNetV2': results_df[results_df['simple_name'].str.contains('MobileV2')],
        'InceptionV3': results_df[results_df['simple_name'].str.contains('InceptV3')],
        'Ensembles': results_df[results_df['simple_name'].str.contains('Ens')]
    }
    
    arch_names = []
    val_maes = []
    colors_list = []
    
    color_map = {'DenseNet121': '#3498db', 'MobileNetV2': '#2ecc71', 
                 'InceptionV3': '#9b59b6', 'Ensembles': '#e74c3c'}
    
    for arch_name, group in arch_groups.items():
        if len(group) > 0:
            for _, row in group.iterrows():
                arch_names.append(row['simple_name'])
                val_maes.append(row['val_mae'])
                colors_list.append(color_map[arch_name])
    
    x_pos = np.arange(len(arch_names))
    bars = plt.bar(x_pos, val_maes, color=colors_list, edgecolor='black', linewidth=1.5, alpha=0.8)
    
    # Add value labels
    for bar, val in zip(bars, val_maes):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.xlabel('Model', fontsize=14, fontweight='bold')
    plt.ylabel('Validation MAE (kg)', fontsize=14, fontweight='bold')
    plt.title('Performance Comparison by Architecture Type', fontsize=16, fontweight='bold', pad=20)
    plt.xticks(x_pos, arch_names, rotation=45, ha='right', fontsize=11)
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=color, edgecolor='black', label=name, alpha=0.8) 
                      for name, color in color_map.items()]
    plt.legend(handles=legend_elements, fontsize=12, loc='upper right')
    
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    filename = 'architecture_comparison.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

# ==================== Dataset Plotting Functions ====================

def plot_weight_distribution(weights, title_suffix=''):
    """Plot weight distribution histogram. Returns BytesIO and filename."""
    plt.figure(figsize=(10, 8))
    
    plt.hist(weights, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    
    mean_weight = weights.mean()
    median_weight = np.median(weights)
    std_weight = weights.std()
    
    plt.axvline(x=mean_weight, color='red', linestyle='--', lw=2, label=f'Mean: {mean_weight:.2f} kg')
    plt.axvline(x=median_weight, color='green', linestyle='--', lw=2, label=f'Median: {median_weight:.2f} kg')
    
    plt.xlabel('Weight (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Frequency', fontsize=14, fontweight='bold')
    plt.title(f'Cattle Weight Distribution{title_suffix}\nStd: {std_weight:.2f} kg, Range: [{weights.min():.2f}, {weights.max():.2f}] kg', 
              fontsize=14, fontweight='bold', pad=20)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    filename = f'dataset_weight_distribution{title_suffix.lower().replace(" ", "_").replace("(", "").replace(")", "")}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_weight_boxplot(y_train, y_val):
    """Plot weight distribution boxplot for train and validation. Returns BytesIO and filename."""
    plt.figure(figsize=(10, 8))
    
    data = [y_train, y_val]
    labels = ['Training Set', 'Validation Set']
    
    bp = plt.boxplot(data, labels=labels, patch_artist=True, showmeans=True,
                     meanprops=dict(marker='D', markerfacecolor='red', markersize=10))
    
    # Color the boxes
    colors = ['lightgreen', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    plt.ylabel('Weight (kg)', fontsize=14, fontweight='bold')
    plt.title('Weight Distribution Comparison: Training vs Validation', fontsize=14, fontweight='bold', pad=20)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    filename = 'dataset_weight_boxplot_train_val.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_dataset_statistics(y_train, y_val):
    """Plot dataset statistics table. Returns BytesIO and filename."""
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('tight')
    ax.axis('off')
    
    stats_data = [
        ['Statistic', 'Training Set', 'Validation Set', 'Overall'],
        ['Count', f'{len(y_train)}', f'{len(y_val)}', f'{len(y_train) + len(y_val)}'],
        ['Mean (kg)', f'{y_train.mean():.2f}', f'{y_val.mean():.2f}', f'{np.concatenate([y_train, y_val]).mean():.2f}'],
        ['Median (kg)', f'{np.median(y_train):.2f}', f'{np.median(y_val):.2f}', f'{np.median(np.concatenate([y_train, y_val])):.2f}'],
        ['Std Dev (kg)', f'{y_train.std():.2f}', f'{y_val.std():.2f}', f'{np.concatenate([y_train, y_val]).std():.2f}'],
        ['Min (kg)', f'{y_train.min():.2f}', f'{y_val.min():.2f}', f'{min(y_train.min(), y_val.min()):.2f}'],
        ['Max (kg)', f'{y_train.max():.2f}', f'{y_val.max():.2f}', f'{max(y_train.max(), y_val.max()):.2f}'],
        ['Q1 (kg)', f'{np.percentile(y_train, 25):.2f}', f'{np.percentile(y_val, 25):.2f}', 
         f'{np.percentile(np.concatenate([y_train, y_val]), 25):.2f}'],
        ['Q3 (kg)', f'{np.percentile(y_train, 75):.2f}', f'{np.percentile(y_val, 75):.2f}', 
         f'{np.percentile(np.concatenate([y_train, y_val]), 75):.2f}'],
    ]
    
    table = ax.table(cellText=stats_data, cellLoc='center', loc='center',
                     colWidths=[0.25, 0.25, 0.25, 0.25])
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 2.5)
    
    # Style header row
    for i in range(4):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Alternate row colors
    for i in range(1, len(stats_data)):
        for j in range(4):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
            else:
                table[(i, j)].set_facecolor('#FFFFFF')
    
    plt.title('Dataset Statistics Summary', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    
    filename = 'dataset_statistics_table.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_train_val_split_visualization(y_train, y_val):
    """Visualize train/validation split. Returns BytesIO and filename."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Pie chart
    sizes = [len(y_train), len(y_val)]
    labels = [f'Training\n({len(y_train)} samples)', f'Validation\n({len(y_val)} samples)']
    colors = ['#2ecc71', '#e74c3c']
    explode = (0.05, 0.05)
    
    axes[0].pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
                shadow=True, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
    axes[0].set_title('Train/Validation Split', fontsize=14, fontweight='bold', pad=15)
    
    # Bar chart
    x_pos = np.arange(2)
    bars = axes[1].bar(x_pos, sizes, color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)
    
    # Add value labels
    for bar, size in zip(bars, sizes):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{size}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    axes[1].set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
    axes[1].set_title('Sample Count Comparison', fontsize=14, fontweight='bold', pad=15)
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(['Training', 'Validation'], fontsize=11)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    filename = 'dataset_train_val_split.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_sample_images_visualization(X, y, num_samples=4):
    """Visualize sample images from dataset showing 4 views. Returns BytesIO and filename."""
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    # Select random samples
    indices = np.random.choice(len(X), num_samples, replace=False)
    
    view_names = ['Front View', 'Back View', 'Left View', 'Right View']
    
    for i, idx in enumerate(indices):
        sample = X[idx]
        weight = y[idx]
        
        # Split 12 channels into 4 RGB images
        front = sample[:, :, 0:3]
        back = sample[:, :, 3:6]
        left = sample[:, :, 6:9]
        right = sample[:, :, 9:12]
        
        views = [front, back, left, right]
        
        for j, (view, name) in enumerate(zip(views, view_names)):
            axes[i, j].imshow(view)
            axes[i, j].axis('off')
            if i == 0:
                axes[i, j].set_title(name, fontsize=12, fontweight='bold', pad=10)
            if j == 0:
                axes[i, j].text(-20, sample.shape[0]/2, f'Sample {i+1}\nWeight: {weight:.1f} kg', 
                              rotation=90, va='center', ha='right', fontsize=11, fontweight='bold')
    
    plt.suptitle('Sample Cattle Images (4-View Dataset)', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    
    filename = 'dataset_sample_images.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_weight_range_distribution(y_train, y_val):
    """Plot weight distribution across ranges. Returns BytesIO and filename."""
    plt.figure(figsize=(12, 8))
    
    # Define weight ranges
    bins = [0, 100, 200, 300, 400, 500, 600, 700, 800]
    labels = ['0-100', '100-200', '200-300', '300-400', '400-500', '500-600', '600-700', '700-800']
    
    # Count samples in each range
    train_counts = np.histogram(y_train, bins=bins)[0]
    val_counts = np.histogram(y_val, bins=bins)[0]
    
    x_pos = np.arange(len(labels))
    width = 0.35
    
    bars1 = plt.bar(x_pos - width/2, train_counts, width, label='Training', 
                    color='#2ecc71', edgecolor='black', linewidth=1.5, alpha=0.8)
    bars2 = plt.bar(x_pos + width/2, val_counts, width, label='Validation', 
                    color='#e74c3c', edgecolor='black', linewidth=1.5, alpha=0.8)
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    plt.xlabel('Weight Range (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Number of Samples', fontsize=14, fontweight='bold')
    plt.title('Sample Distribution Across Weight Ranges', fontsize=16, fontweight='bold', pad=20)
    plt.xticks(x_pos, labels, rotation=45, ha='right', fontsize=11)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    filename = 'dataset_weight_range_distribution.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

# ==================== NEW ADVANCED PLOTTING FUNCTIONS ====================

def plot_training_history(history, model_name):
    """Plot training history curves. Returns BytesIO and filename."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    epochs = range(1, len(history['loss']) + 1)
    
    # Loss
    axes[0, 0].plot(epochs, history['loss'], 'b-', linewidth=2, label='Training Loss')
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Validation Loss')
    axes[0, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[0, 0].set_ylabel('Loss (MSE)', fontsize=12, fontweight='bold')
    axes[0, 0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
    axes[0, 0].legend(fontsize=11)
    axes[0, 0].grid(True, alpha=0.3)
    
    # MAE
    axes[0, 1].plot(epochs, history['mae'], 'b-', linewidth=2, label='Training MAE')
    axes[0, 1].plot(epochs, history['val_mae'], 'r-', linewidth=2, label='Validation MAE')
    axes[0, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[0, 1].set_ylabel('MAE (kg)', fontsize=12, fontweight='bold')
    axes[0, 1].set_title('Training & Validation MAE', fontsize=14, fontweight='bold')
    axes[0, 1].legend(fontsize=11)
    axes[0, 1].grid(True, alpha=0.3)
    
    # MSE
    axes[1, 0].plot(epochs, history['mse'], 'b-', linewidth=2, label='Training MSE')
    axes[1, 0].plot(epochs, history['val_mse'], 'r-', linewidth=2, label='Validation MSE')
    axes[1, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('MSE', fontsize=12, fontweight='bold')
    axes[1, 0].set_title('Training & Validation MSE', fontsize=14, fontweight='bold')
    axes[1, 0].legend(fontsize=11)
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning rate (if available) or loss difference
    if 'lr' in history:
        axes[1, 1].plot(epochs, history['lr'], 'g-', linewidth=2)
        axes[1, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
        axes[1, 1].set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
        axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
        axes[1, 1].set_yscale('log')
        axes[1, 1].grid(True, alpha=0.3)
    else:
        # Plot loss difference if no LR
        loss_diff = [abs(t - v) for t, v in zip(history['loss'], history['val_loss'])]
        axes[1, 1].plot(epochs, loss_diff, 'purple', linewidth=2)
        axes[1, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
        axes[1, 1].set_ylabel('|Train Loss - Val Loss|', fontsize=12, fontweight='bold')
        axes[1, 1].set_title('Train-Val Loss Gap', fontsize=14, fontweight='bold')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(f'{model_name} - Training History', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_training_history.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_confusion_matrix_weight_ranges(y_true, y_pred, model_name, dataset_type):
    """Plot confusion matrix for weight ranges. Returns BytesIO and filename."""
    # Define weight bins
    bins = [0, 200, 300, 400, 500, 800]
    labels = ['0-200', '200-300', '300-400', '400-500', '500+']
    
    # Bin the true and predicted values
    y_true_binned = np.digitize(y_true, bins) - 1
    y_pred_binned = np.digitize(y_pred, bins) - 1
    
    # Create confusion matrix
    n_bins = len(labels)
    conf_matrix = np.zeros((n_bins, n_bins))
    
    for true_bin, pred_bin in zip(y_true_binned, y_pred_binned):
        if 0 <= true_bin < n_bins and 0 <= pred_bin < n_bins:
            conf_matrix[true_bin, pred_bin] += 1
    
    # Normalize by row (true labels)
    row_sums = conf_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # Avoid division by zero
    conf_matrix_norm = conf_matrix / row_sums * 100
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix_norm, annot=True, fmt='.1f', cmap='Blues', 
                xticklabels=labels, yticklabels=labels,
                cbar_kws={'label': 'Percentage (%)'}, linewidths=0.5, linecolor='black')
    
    plt.xlabel('Predicted Weight Range (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Actual Weight Range (kg)', fontsize=14, fontweight='bold')
    plt.title(f'{model_name} - Weight Range Confusion Matrix ({dataset_type})\nValues are row-wise percentages', 
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_confusion_matrix_{dataset_type.lower()}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_percentage_error_analysis(y_true, y_pred, model_name, dataset_type):
    """Plot percentage error vs actual weights. Returns BytesIO and filename."""
    percentage_errors = ((y_pred - y_true) / y_true) * 100
    
    plt.figure(figsize=(12, 8))
    
    # Scatter plot
    scatter = plt.scatter(y_true, percentage_errors, alpha=0.6, s=50, 
                         c=np.abs(percentage_errors), cmap='YlOrRd', 
                         edgecolors='k', linewidth=0.5)
    plt.colorbar(scatter, label='Absolute % Error')
    
    # Reference lines
    plt.axhline(y=0, color='green', linestyle='--', lw=2, label='Perfect Prediction')
    plt.axhline(y=5, color='orange', linestyle=':', lw=1.5, alpha=0.7, label='±5% Error')
    plt.axhline(y=-5, color='orange', linestyle=':', lw=1.5, alpha=0.7)
    plt.axhline(y=10, color='red', linestyle=':', lw=1.5, alpha=0.7, label='±10% Error')
    plt.axhline(y=-10, color='red', linestyle=':', lw=1.5, alpha=0.7)
    
    # Statistics
    mean_pe = percentage_errors.mean()
    median_pe = np.median(percentage_errors)
    
    plt.xlabel('Actual Weight (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Percentage Error (%)', fontsize=14, fontweight='bold')
    plt.title(f'{model_name} - Percentage Error Analysis ({dataset_type})\nMean: {mean_pe:.2f}%, Median: {median_pe:.2f}%', 
              fontsize=14, fontweight='bold', pad=20)
    plt.legend(fontsize=11, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_percentage_error_{dataset_type.lower()}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_cumulative_error_distribution(y_true, y_pred, model_name, dataset_type):
    """Plot cumulative distribution of errors. Returns BytesIO and filename."""
    abs_errors = np.abs(y_true - y_pred)
    sorted_errors = np.sort(abs_errors)
    cumulative = np.arange(1, len(sorted_errors) + 1) / len(sorted_errors) * 100
    
    plt.figure(figsize=(12, 8))
    plt.plot(sorted_errors, cumulative, linewidth=2.5, color='steelblue')
    
    # Mark key percentiles
    percentiles = [50, 75, 90, 95]
    colors_p = ['green', 'orange', 'red', 'darkred']
    
    for p, color in zip(percentiles, colors_p):
        error_at_p = np.percentile(abs_errors, p)
        plt.axvline(x=error_at_p, color=color, linestyle='--', lw=1.5, alpha=0.7)
        plt.axhline(y=p, color=color, linestyle='--', lw=1.5, alpha=0.7)
        plt.plot(error_at_p, p, 'o', color=color, markersize=10, 
                label=f'{p}% within {error_at_p:.2f} kg')
    
    plt.xlabel('Absolute Error (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Cumulative Percentage (%)', fontsize=14, fontweight='bold')
    plt.title(f'{model_name} - Cumulative Error Distribution ({dataset_type})', 
              fontsize=14, fontweight='bold', pad=20)
    plt.legend(fontsize=11, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.xlim(left=0)
    plt.ylim([0, 100])
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_cumulative_error_{dataset_type.lower()}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_bland_altman(y_true, y_pred, model_name, dataset_type):
    """Plot Bland-Altman plot for agreement analysis. Returns BytesIO and filename."""
    mean_values = (y_true + y_pred) / 2
    differences = y_pred - y_true
    
    mean_diff = differences.mean()
    std_diff = differences.std()
    
    plt.figure(figsize=(12, 8))
    
    plt.scatter(mean_values, differences, alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
    
    # Mean difference line
    plt.axhline(y=mean_diff, color='blue', linestyle='-', lw=2, label=f'Mean Diff: {mean_diff:.2f} kg')
    
    # Limits of agreement (±1.96 SD)
    upper_loa = mean_diff + 1.96 * std_diff
    lower_loa = mean_diff - 1.96 * std_diff
    
    plt.axhline(y=upper_loa, color='red', linestyle='--', lw=2, label=f'Upper LoA: {upper_loa:.2f} kg')
    plt.axhline(y=lower_loa, color='red', linestyle='--', lw=2, label=f'Lower LoA: {lower_loa:.2f} kg')
    
    # Fill between limits
    plt.fill_between([mean_values.min(), mean_values.max()], lower_loa, upper_loa, alpha=0.2, color='red')
    
    plt.xlabel('Mean of Actual and Predicted (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Difference (Predicted - Actual) (kg)', fontsize=14, fontweight='bold')
    plt.title(f'{model_name} - Bland-Altman Plot ({dataset_type})\nStd Dev: {std_diff:.2f} kg', 
              fontsize=14, fontweight='bold', pad=20)
    plt.legend(fontsize=11, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_bland_altman_{dataset_type.lower()}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_model_calibration(y_true, y_pred, model_name, dataset_type):
    """Plot calibration curve showing if model predictions are well-calibrated. Returns BytesIO and filename."""
    # Divide into bins
    n_bins = 10
    bins = np.linspace(y_true.min(), y_true.max(), n_bins + 1)
    
    bin_centers = []
    mean_true = []
    mean_pred = []
    bin_counts = []
    
    for i in range(n_bins):
        mask = (y_true >= bins[i]) & (y_true < bins[i + 1])
        if i == n_bins - 1:  # Include right edge for last bin
            mask = (y_true >= bins[i]) & (y_true <= bins[i + 1])
        
        if mask.sum() > 0:
            bin_centers.append((bins[i] + bins[i + 1]) / 2)
            mean_true.append(y_true[mask].mean())
            mean_pred.append(y_pred[mask].mean())
            bin_counts.append(mask.sum())
    
    plt.figure(figsize=(10, 10))
    
    # Scatter plot with size based on count
    sizes = [count * 5 for count in bin_counts]
    plt.scatter(mean_true, mean_pred, s=sizes, alpha=0.6, edgecolors='k', linewidth=2)
    
    # Perfect calibration line
    min_val = min(min(mean_true), min(mean_pred))
    max_val = max(max(mean_true), max(mean_pred))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Calibration')
    
    # Add labels for each bin
    for mt, mp, count in zip(mean_true, mean_pred, bin_counts):
        plt.annotate(f'n={count}', (mt, mp), xytext=(5, 5), textcoords='offset points', 
                    fontsize=9, alpha=0.7)
    
    plt.xlabel('Mean Actual Weight per Bin (kg)', fontsize=14, fontweight='bold')
    plt.ylabel('Mean Predicted Weight per Bin (kg)', fontsize=14, fontweight='bold')
    plt.title(f'{model_name} - Calibration Plot ({dataset_type})\n(Bubble size = sample count)', 
              fontsize=14, fontweight='bold', pad=20)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_calibration_{dataset_type.lower()}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_error_by_weight_range(y_true, y_pred, model_name, dataset_type):
    """Plot error statistics grouped by weight ranges. Returns BytesIO and filename."""
    # Define weight ranges
    bins = [0, 200, 300, 400, 500, 800]
    labels = ['0-200', '200-300', '300-400', '400-500', '500+']
    
    errors = y_pred - y_true
    abs_errors = np.abs(errors)
    
    # Group errors by weight range
    grouped_errors = []
    grouped_abs_errors = []
    counts = []
    
    for i in range(len(bins) - 1):
        if i < len(bins) - 2:
            mask = (y_true >= bins[i]) & (y_true < bins[i + 1])
        else:
            mask = (y_true >= bins[i])
        
        if mask.sum() > 0:
            grouped_errors.append(errors[mask])
            grouped_abs_errors.append(abs_errors[mask])
            counts.append(mask.sum())
        else:
            grouped_errors.append(np.array([]))
            grouped_abs_errors.append(np.array([]))
            counts.append(0)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Box plot of errors
    valid_errors = [e for e in grouped_errors if len(e) > 0]
    valid_labels = [l for l, e in zip(labels, grouped_errors) if len(e) > 0]
    
    bp1 = axes[0, 0].boxplot(valid_errors, labels=valid_labels, patch_artist=True, showmeans=True)
    for patch in bp1['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    axes[0, 0].axhline(y=0, color='red', linestyle='--', lw=2)
    axes[0, 0].set_xlabel('Weight Range (kg)', fontsize=12, fontweight='bold')
    axes[0, 0].set_ylabel('Error (kg)', fontsize=12, fontweight='bold')
    axes[0, 0].set_title('Error Distribution by Weight Range', fontsize=13, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Bar plot of MAE by range
    mae_by_range = [e.mean() if len(e) > 0 else 0 for e in grouped_abs_errors]
    x_pos = np.arange(len(labels))
    bars = axes[0, 1].bar(x_pos, mae_by_range, color='coral', edgecolor='black', linewidth=1.5, alpha=0.8)
    
    for bar, mae, count in zip(bars, mae_by_range, counts):
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{mae:.2f}\n(n={count})', ha='center', va='bottom', 
                       fontsize=10, fontweight='bold')
    
    axes[0, 1].set_xlabel('Weight Range (kg)', fontsize=12, fontweight='bold')
    axes[0, 1].set_ylabel('Mean Absolute Error (kg)', fontsize=12, fontweight='bold')
    axes[0, 1].set_title('MAE by Weight Range', fontsize=13, fontweight='bold')
    axes[0, 1].set_xticks(x_pos)
    axes[0, 1].set_xticklabels(labels)
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # Std dev by range
    std_by_range = [e.std() if len(e) > 0 else 0 for e in grouped_abs_errors]
    bars = axes[1, 0].bar(x_pos, std_by_range, color='skyblue', edgecolor='black', linewidth=1.5, alpha=0.8)
    
    for bar, std in zip(bars, std_by_range):
        height = bar.get_height()
        if height > 0:
            axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                           f'{std:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    axes[1, 0].set_xlabel('Weight Range (kg)', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Error Std Dev (kg)', fontsize=12, fontweight='bold')
    axes[1, 0].set_title('Error Variability by Weight Range', fontsize=13, fontweight='bold')
    axes[1, 0].set_xticks(x_pos)
    axes[1, 0].set_xticklabels(labels)
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Sample count
    bars = axes[1, 1].bar(x_pos, counts, color='lightgreen', edgecolor='black', linewidth=1.5, alpha=0.8)
    
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        if height > 0:
            axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                           f'{count}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    axes[1, 1].set_xlabel('Weight Range (kg)', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
    axes[1, 1].set_title('Sample Distribution', fontsize=13, fontweight='bold')
    axes[1, 1].set_xticks(x_pos)
    axes[1, 1].set_xticklabels(labels)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.suptitle(f'{model_name} - Error Analysis by Weight Range ({dataset_type})', 
                fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    
    filename = f'{model_name.replace(" ", "_").replace(":", "")}_error_by_range_{dataset_type.lower()}.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

def plot_computation_comparison(results_df):
    """Plot training time and inference speed comparison. Returns BytesIO and filename."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    x_pos = np.arange(len(results_df))
    colors = ['#1f77b4' if 'Ens' not in name else '#ff7f0e' for name in results_df['simple_name']]
    
    # Training time
    bars1 = axes[0].bar(x_pos, results_df['training_time'], color=colors, 
                        edgecolor='black', linewidth=1.5, alpha=0.8)
    
    for bar, val in zip(bars1, results_df['training_time']):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.1f}s', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    axes[0].set_xlabel('Model', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Training Time (seconds)', fontsize=12, fontweight='bold')
    axes[0].set_title('Training Time Comparison', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(results_df['simple_name'], rotation=45, ha='right', fontsize=10)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Inference time (validation)
    inference_times_ms = results_df['val_inference_time'] * 1000
    bars2 = axes[1].bar(x_pos, inference_times_ms, color=colors, 
                        edgecolor='black', linewidth=1.5, alpha=0.8)
    
    for bar, val in zip(bars2, inference_times_ms):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.1f}ms', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    axes[1].set_xlabel('Model', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Inference Time (milliseconds)', fontsize=12, fontweight='bold')
    axes[1].set_title('Inference Time Comparison', fontsize=14, fontweight='bold')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(results_df['simple_name'], rotation=45, ha='right', fontsize=10)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Efficiency score (lower MAE / training time)
    efficiency = results_df['val_mae'] / (results_df['training_time'] / 60)  # MAE per minute
    bars3 = axes[2].bar(x_pos, efficiency, color=colors, 
                        edgecolor='black', linewidth=1.5, alpha=0.8)
    
    for bar, val in zip(bars3, efficiency):
        height = bar.get_height()
        axes[2].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    axes[2].set_xlabel('Model', fontsize=12, fontweight='bold')
    axes[2].set_ylabel('MAE / Training Time (kg/min)', fontsize=12, fontweight='bold')
    axes[2].set_title('Efficiency Score\n(Lower is Better)', fontsize=14, fontweight='bold')
    axes[2].set_xticks(x_pos)
    axes[2].set_xticklabels(results_df['simple_name'], rotation=45, ha='right', fontsize=10)
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    filename = 'computation_comparison.png'
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=300, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf, filename

# ==================== Main ====================

def main():
    print("\n" + "="*70)
    print("CATTLE WEIGHT PREDICTION - 6 MODELS + 3 ENSEMBLES")
    print("="*70)
    
    # Check paths
    if not os.path.exists(METADATA_PATH):
        print(f"ERROR: Metadata file not found at {METADATA_PATH}")
        return
    if not os.path.exists(IMAGES_BASE_PATH):
        print(f"ERROR: Images directory not found at {IMAGES_BASE_PATH}")
        return
    
    # Load data
    print("\n" + "="*70)
    print("LOADING DATASET")
    print("="*70)
    X, y, valid_ids = load_dataset(METADATA_PATH, IMAGES_BASE_PATH)
    
    if len(X) == 0:
        print("ERROR: No valid data loaded!")
        return
    
    # Split data
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    print(f"\nTrain samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    
    # ==================== GENERATE DATASET PLOTS ====================
    print("\n" + "="*70)
    print("GENERATING DATASET VISUALIZATION PLOTS")
    print("="*70)
    
    dataset_plot_buffers = []
    
    print("\n1. Weight Distribution Plots...")
    dataset_plot_buffers.append(plot_weight_distribution(y, title_suffix=' (Overall)'))
    dataset_plot_buffers.append(plot_weight_distribution(y_train, title_suffix=' (Training Set)'))
    dataset_plot_buffers.append(plot_weight_distribution(y_val, title_suffix=' (Validation Set)'))
    
    print("2. Weight Comparison Plots...")
    dataset_plot_buffers.append(plot_weight_boxplot(y_train, y_val))
    dataset_plot_buffers.append(plot_weight_range_distribution(y_train, y_val))
    
    print("3. Dataset Statistics...")
    dataset_plot_buffers.append(plot_dataset_statistics(y_train, y_val))
    
    print("4. Train/Validation Split Visualization...")
    dataset_plot_buffers.append(plot_train_val_split_visualization(y_train, y_val))
    
    print("5. Sample Images Visualization...")
    dataset_plot_buffers.append(plot_sample_images_visualization(X, y, num_samples=4))
    
    # Create zip file for dataset plots
    dataset_zip_filename = '/kaggle/working/dataset_plots.zip'
    with zipfile.ZipFile(dataset_zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for buf, filename in dataset_plot_buffers:
            zipf.writestr(filename, buf.getvalue())
    print(f"\n✓ Saved {len(dataset_plot_buffers)} dataset plots to {dataset_zip_filename}")
    
    all_results = []
    
    # Train all 6 models
    print("\n" + "="*70)
    print("TRAINING 6 MODELS")
    print("="*70)
    
    for model_key in MODEL_VERSIONS.keys():
        config = MODEL_VERSIONS[model_key]
        
        clean_memory()
        
        # Build model
        model = build_model(
            config['base_model'],
            X_train.shape[1:],
            model_key,
            config
        )
        
        # Train
        results, trained_model = train_and_evaluate(
            model, X_train, y_train, X_val, y_val,
            model_key,
            config['learning_rate']
        )
        
        all_results.append(results)
        
        # Get predictions for plotting
        print(f"\nGenerating plots for {config['display_name']}...")
        simple_name = SIMPLE_NAMES[model_key]
        
        y_pred_train = trained_model.predict(X_train, verbose=0).flatten()
        y_pred_val = trained_model.predict(X_val, verbose=0).flatten()
        
        # Generate individual model plots and collect them in memory
        plot_buffers = []
        
        # Training history
        plot_buffers.append(plot_training_history(results['history'], simple_name))
        
        # Original plots
        plot_buffers.append(plot_predicted_vs_actual(y_train, y_pred_train, simple_name, 'Train'))
        plot_buffers.append(plot_predicted_vs_actual(y_val, y_pred_val, simple_name, 'Validation'))
        plot_buffers.append(plot_residuals(y_train, y_pred_train, simple_name, 'Train'))
        plot_buffers.append(plot_residuals(y_val, y_pred_val, simple_name, 'Validation'))
        plot_buffers.append(plot_error_distribution(y_train, y_pred_train, simple_name, 'Train'))
        plot_buffers.append(plot_error_distribution(y_val, y_pred_val, simple_name, 'Validation'))
        plot_buffers.append(plot_absolute_errors(y_train, y_pred_train, simple_name, 'Train'))
        plot_buffers.append(plot_absolute_errors(y_val, y_pred_val, simple_name, 'Validation'))
        
        # New advanced plots
        plot_buffers.append(plot_confusion_matrix_weight_ranges(y_train, y_pred_train, simple_name, 'Train'))
        plot_buffers.append(plot_confusion_matrix_weight_ranges(y_val, y_pred_val, simple_name, 'Validation'))
        plot_buffers.append(plot_percentage_error_analysis(y_train, y_pred_train, simple_name, 'Train'))
        plot_buffers.append(plot_percentage_error_analysis(y_val, y_pred_val, simple_name, 'Validation'))
        plot_buffers.append(plot_cumulative_error_distribution(y_train, y_pred_train, simple_name, 'Train'))
        plot_buffers.append(plot_cumulative_error_distribution(y_val, y_pred_val, simple_name, 'Validation'))
        plot_buffers.append(plot_bland_altman(y_train, y_pred_train, simple_name, 'Train'))
        plot_buffers.append(plot_bland_altman(y_val, y_pred_val, simple_name, 'Validation'))
        plot_buffers.append(plot_model_calibration(y_train, y_pred_train, simple_name, 'Train'))
        plot_buffers.append(plot_model_calibration(y_val, y_pred_val, simple_name, 'Validation'))
        plot_buffers.append(plot_error_by_weight_range(y_train, y_pred_train, simple_name, 'Train'))
        plot_buffers.append(plot_error_by_weight_range(y_val, y_pred_val, simple_name, 'Validation'))
        
        # Create zip file for this model's plots
        zip_filename = f'/kaggle/working/{simple_name.replace(" ", "_").replace(":", "")}_plots.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for buf, filename in plot_buffers:
                zipf.writestr(filename, buf.getvalue())
        print(f"  ✓ Saved {len(plot_buffers)} plots to {zip_filename}")
        
        # Save model in native Keras format
        model_path = f'/kaggle/working/{model_key}.keras'
        trained_model.save(model_path)
        print(f"Model saved to {model_path}")
    
    # ==================== ENSEMBLE PREDICTIONS ====================
    print("\n" + "="*70)
    print("ENSEMBLE MODEL EVALUATION")
    print("="*70)
    
    # Define ensemble combinations
    ensemble_configs = [
        {
            'name': 'Ensemble 1: DenseNet121 v1+v2',
            'models': ['DenseNet121_v1_original', 'DenseNet121_v2_finetune']
        },
        {
            'name': 'Ensemble 2: DenseNet121 v2 + MobileNetV2 v2',
            'models': ['DenseNet121_v2_finetune', 'MobileNetV2_v2_finetune']
        },
        {
            'name': 'Ensemble 3: DenseNet121 v2 + InceptionV3 v2',
            'models': ['DenseNet121_v2_finetune', 'InceptionV3_v2_finetune']
        }
    ]
    
    # Evaluate ensembles
    all_val_errors = []  # Store errors for boxplot
    
    # First, collect individual model predictions and errors
    for result in all_results:
        model_key = result['model_name']
        model_path = f'/kaggle/working/{model_key}.keras'
        try:
            # Load model with custom objects
            model = keras.models.load_model(model_path, custom_objects={'SplitChannels': SplitChannels})
            
            val_pred = model.predict(X_val, verbose=0).flatten()
            val_errors = y_val - val_pred
            
            simple_name = SIMPLE_NAMES[model_key]
            all_val_errors.append((simple_name, val_errors))
            
            del model
            keras.backend.clear_session()
        except Exception as e:
            print(f"  ERROR loading model {model_key}: {e}")
            continue
    
    for ensemble_config in ensemble_configs:
        print(f"\nEvaluating {ensemble_config['name']}...")
        
        # Load models and make predictions
        train_preds = []
        val_preds = []
        
        for model_key in ensemble_config['models']:
            model_path = f'/kaggle/working/{model_key}.keras'
            print(f"  Loading {model_key}...")
            try:
                # Load model with custom objects
                model = keras.models.load_model(model_path, custom_objects={'SplitChannels': SplitChannels})
                
                train_pred = model.predict(X_train, verbose=0).flatten()
                val_pred = model.predict(X_val, verbose=0).flatten()
                
                train_preds.append(train_pred)
                val_preds.append(val_pred)
                
                # Clean up model from memory
                del model
                keras.backend.clear_session()
                gc.collect()  # Force garbage collection
            except Exception as e:
                print(f"  ERROR loading model {model_key}: {e}")
                continue
        
        # Skip if not enough models loaded
        if len(train_preds) < len(ensemble_config['models']):
            print(f"  WARNING: Only {len(train_preds)}/{len(ensemble_config['models'])} models loaded. Skipping ensemble.")
            continue
        
        # Average predictions
        ensemble_train_pred = np.mean(train_preds, axis=0)
        ensemble_val_pred = np.mean(val_preds, axis=0)
        
        # Calculate metrics
        train_mae = mean_absolute_error(y_train, ensemble_train_pred)
        train_rmse = np.sqrt(mean_squared_error(y_train, ensemble_train_pred))
        train_r2 = r2_score(y_train, ensemble_train_pred)
        
        val_mae = mean_absolute_error(y_val, ensemble_val_pred)
        val_rmse = np.sqrt(mean_squared_error(y_val, ensemble_val_pred))
        val_r2 = r2_score(y_val, ensemble_val_pred)
        
        print(f"\n{ensemble_config['name']} Results:")
        print(f"Train - MAE: {train_mae:.2f}, RMSE: {train_rmse:.2f}, R²: {train_r2:.4f}")
        print(f"Val   - MAE: {val_mae:.2f}, RMSE: {val_rmse:.2f}, R²: {val_r2:.4f}")
        
        # Generate ensemble plots
        print(f"\nGenerating plots for {ensemble_config['name']}...")
        simple_name = SIMPLE_NAMES[ensemble_config['name']]
        
        # Collect ensemble plots
        ensemble_plot_buffers = []
        
        # Original plots
        ensemble_plot_buffers.append(plot_predicted_vs_actual(y_train, ensemble_train_pred, simple_name, 'Train'))
        ensemble_plot_buffers.append(plot_predicted_vs_actual(y_val, ensemble_val_pred, simple_name, 'Validation'))
        ensemble_plot_buffers.append(plot_residuals(y_train, ensemble_train_pred, simple_name, 'Train'))
        ensemble_plot_buffers.append(plot_residuals(y_val, ensemble_val_pred, simple_name, 'Validation'))
        ensemble_plot_buffers.append(plot_error_distribution(y_train, ensemble_train_pred, simple_name, 'Train'))
        ensemble_plot_buffers.append(plot_error_distribution(y_val, ensemble_val_pred, simple_name, 'Validation'))
        ensemble_plot_buffers.append(plot_absolute_errors(y_train, ensemble_train_pred, simple_name, 'Train'))
        ensemble_plot_buffers.append(plot_absolute_errors(y_val, ensemble_val_pred, simple_name, 'Validation'))
        
        # New advanced plots
        ensemble_plot_buffers.append(plot_confusion_matrix_weight_ranges(y_train, ensemble_train_pred, simple_name, 'Train'))
        ensemble_plot_buffers.append(plot_confusion_matrix_weight_ranges(y_val, ensemble_val_pred, simple_name, 'Validation'))
        ensemble_plot_buffers.append(plot_percentage_error_analysis(y_train, ensemble_train_pred, simple_name, 'Train'))
        ensemble_plot_buffers.append(plot_percentage_error_analysis(y_val, ensemble_val_pred, simple_name, 'Validation'))
        ensemble_plot_buffers.append(plot_cumulative_error_distribution(y_train, ensemble_train_pred, simple_name, 'Train'))
        ensemble_plot_buffers.append(plot_cumulative_error_distribution(y_val, ensemble_val_pred, simple_name, 'Validation'))
        ensemble_plot_buffers.append(plot_bland_altman(y_train, ensemble_train_pred, simple_name, 'Train'))
        ensemble_plot_buffers.append(plot_bland_altman(y_val, ensemble_val_pred, simple_name, 'Validation'))
        ensemble_plot_buffers.append(plot_model_calibration(y_train, ensemble_train_pred, simple_name, 'Train'))
        ensemble_plot_buffers.append(plot_model_calibration(y_val, ensemble_val_pred, simple_name, 'Validation'))
        ensemble_plot_buffers.append(plot_error_by_weight_range(y_train, ensemble_train_pred, simple_name, 'Train'))
        ensemble_plot_buffers.append(plot_error_by_weight_range(y_val, ensemble_val_pred, simple_name, 'Validation'))
        
        # Create zip file for this ensemble's plots
        zip_filename = f'/kaggle/working/{simple_name.replace(" ", "_").replace(":", "")}_plots.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for buf, filename in ensemble_plot_buffers:
                zipf.writestr(filename, buf.getvalue())
        print(f"  ✓ Saved {len(ensemble_plot_buffers)} plots to {zip_filename}")
        
        # Store errors for comparison
        val_errors_ensemble = y_val - ensemble_val_pred
        all_val_errors.append((simple_name, val_errors_ensemble))
        
        # Add to results
        ensemble_result = {
            'model_name': ensemble_config['name'],
            'display_name': ensemble_config['name'],
            'train_mae': train_mae,
            'train_rmse': train_rmse,
            'train_r2': train_r2,
            'val_mae': val_mae,
            'val_rmse': val_rmse,
            'val_r2': val_r2,
            'epochs_trained': '-'
        }
        all_results.append(ensemble_result)
    
    # ==================== COMPARISON TABLE ====================
    print("\n" + "="*70)
    print("MODEL COMPARISON RESULTS (Including Ensembles)")
    print("="*70)
    
    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values('val_mae')
    
    # Create display table
    display_df = pd.DataFrame({
        'Model': results_df['display_name'],
        'Train MAE': results_df['train_mae'].round(2),
        'Train RMSE': results_df['train_rmse'].round(2),
        'Train R²': results_df['train_r2'].round(6),
        'Val MAE': results_df['val_mae'].round(2),
        'Val RMSE': results_df['val_rmse'].round(2),
        'Val R²': results_df['val_r2'].round(6),
        'Epochs': results_df['epochs_trained']
    })
    
    print("\n" + display_df.to_string(index=False))
    
    # Generate comparison plots
    print("\n" + "="*70)
    print("GENERATING COMPARISON PLOTS")
    print("="*70)
    
    # Add simple names to results_df for plotting
    results_df['simple_name'] = results_df['model_name'].map(SIMPLE_NAMES)
    
    # Collect all comparison plots
    comparison_plot_buffers = []
    
    # Metrics comparison plots
    print("\n1. Metrics Comparison Plots...")
    comparison_plot_buffers.append(plot_metrics_comparison(results_df, 'val_mae', 'Validation MAE (kg)', 
                           'Validation MAE Comparison Across All Models', 'val_mae_comparison.png'))
    comparison_plot_buffers.append(plot_metrics_comparison(results_df, 'val_rmse', 'Validation RMSE (kg)', 
                           'Validation RMSE Comparison Across All Models', 'val_rmse_comparison.png'))
    comparison_plot_buffers.append(plot_metrics_comparison(results_df, 'val_r2', 'Validation R² Score', 
                           'Validation R² Comparison Across All Models', 'val_r2_comparison.png'))
    
    # Train vs Val comparison plots
    print("\n2. Training vs Validation Comparison Plots...")
    comparison_plot_buffers.append(plot_train_val_comparison(results_df, 'train_mae', 'val_mae', 'MAE (kg)',
                             'Training vs Validation MAE', 'train_val_mae_comparison.png'))
    comparison_plot_buffers.append(plot_train_val_comparison(results_df, 'train_rmse', 'val_rmse', 'RMSE (kg)',
                             'Training vs Validation RMSE', 'train_val_rmse_comparison.png'))
    comparison_plot_buffers.append(plot_train_val_comparison(results_df, 'train_r2', 'val_r2', 'R² Score',
                             'Training vs Validation R²', 'train_val_r2_comparison.png'))
    
    # Additional analysis plots
    print("\n3. Additional Analysis Plots...")
    comparison_plot_buffers.append(plot_all_metrics_heatmap(results_df))
    comparison_plot_buffers.append(plot_error_boxplot(all_val_errors))
    
    ens_buf, ens_fname = plot_ensemble_improvement(results_df)
    if ens_buf is not None:
        comparison_plot_buffers.append((ens_buf, ens_fname))
    comparison_plot_buffers.append(plot_model_ranking(results_df))
    comparison_plot_buffers.append(plot_r2_comparison_scatter(results_df))
    comparison_plot_buffers.append(plot_overfitting_analysis(results_df))
    comparison_plot_buffers.append(plot_architecture_comparison(results_df))
    
    # Computation comparison (NEW)
    print("\n4. Computation & Efficiency Comparison...")
    comparison_plot_buffers.append(plot_computation_comparison(results_df))
    
    # Create zip file for comparison plots
    comparison_zip_filename = '/kaggle/working/comparison_plots.zip'
    with zipfile.ZipFile(comparison_zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for buf, filename in comparison_plot_buffers:
            zipf.writestr(filename, buf.getvalue())
    print(f"\n✓ Saved {len(comparison_plot_buffers)} comparison plots to {comparison_zip_filename}")
    
    print("\n✓ All plots generated and saved to zip files!")
    total_dataset = 8  # dataset plots
    total_individual = 6 * 23  # 6 individual models × 23 plots each (1 history + 8 original + 14 new)
    total_ensemble = 3 * 22    # 3 ensemble models × 22 plots each (8 original + 14 new, no history)
    total_comparison = len(comparison_plot_buffers)     # comparison plots
    total_plots = total_dataset + total_individual + total_ensemble + total_comparison
    print(f"Total plots created: {total_plots}")
    print(f"  - Dataset Plots: {total_dataset} plots - saved in 1 zip file")
    print(f"  - Individual Models: {total_individual} plots (6 models × 23 plots) - saved in 6 zip files")
    print(f"  - Ensemble Models: {total_ensemble} plots (3 ensembles × 22 plots) - saved in 3 zip files")
    print(f"  - Comparison Plots: {total_comparison} plots - saved in 1 zip file")
    
    # ==================== DISPLAY ALL PLOTS SUMMARY ====================
    print("\n" + "="*70)
    print("ALL PLOTS SAVED IN ZIP FILES")
    print("="*70)
    
    print("\n📦 ZIP FILES CREATED:")
    
    print("\n  📊 DATASET PLOTS (1 zip file, 8 plots):")
    print("    ✓ dataset_plots.zip")
    print("       - Weight Distribution (Overall)")
    print("       - Weight Distribution (Training Set)")
    print("       - Weight Distribution (Validation Set)")
    print("       - Weight Boxplot (Train vs Val)")
    print("       - Weight Range Distribution")
    print("       - Dataset Statistics Table")
    print("       - Train/Validation Split Visualization")
    print("       - Sample Images (4-View Visualization)")
    
    print("\n  📊 INDIVIDUAL MODEL PLOTS (6 zip files, 23 plots each = 138 total):")
    for model_key in MODEL_VERSIONS.keys():
        simple_name = SIMPLE_NAMES[model_key]
        zip_name = f'{simple_name.replace(" ", "_").replace(":", "")}_plots.zip'
        print(f"    ✓ {zip_name}")
        print(f"       - Training History (Loss, MAE, MSE curves)")
        print(f"       - Predicted vs Actual (Train & Validation)")
        print(f"       - Residual Plots (Train & Validation)")
        print(f"       - Error Distribution (Train & Validation)")
        print(f"       - Absolute Errors (Train & Validation)")
        print(f"       - Confusion Matrix for Weight Ranges (Train & Validation)")
        print(f"       - Percentage Error Analysis (Train & Validation)")
        print(f"       - Cumulative Error Distribution (Train & Validation)")
        print(f"       - Bland-Altman Plots (Train & Validation)")
        print(f"       - Model Calibration (Train & Validation)")
        print(f"       - Error Analysis by Weight Range (Train & Validation)")
    
    print("\n  📊 ENSEMBLE MODEL PLOTS (3 zip files, 22 plots each = 66 total):")
    for ens_key in ['Ensemble 1: DenseNet121 v1+v2', 
                    'Ensemble 2: DenseNet121 v2 + MobileNetV2 v2',
                    'Ensemble 3: DenseNet121 v2 + InceptionV3 v2']:
        simple_name = SIMPLE_NAMES[ens_key]
        zip_name = f'{simple_name.replace(" ", "_").replace(":", "")}_plots.zip'
        print(f"    ✓ {zip_name}")
        print(f"       - Predicted vs Actual (Train & Validation)")
        print(f"       - Residual Plots (Train & Validation)")
        print(f"       - Error Distribution (Train & Validation)")
        print(f"       - Absolute Errors (Train & Validation)")
        print(f"       - Confusion Matrix for Weight Ranges (Train & Validation)")
        print(f"       - Percentage Error Analysis (Train & Validation)")
        print(f"       - Cumulative Error Distribution (Train & Validation)")
        print(f"       - Bland-Altman Plots (Train & Validation)")
        print(f"       - Model Calibration (Train & Validation)")
        print(f"       - Error Analysis by Weight Range (Train & Validation)")
    
    print(f"\n  📊 COMPARISON & ANALYSIS PLOTS (1 zip file, {total_comparison} plots):")
    print("    ✓ comparison_plots.zip")
    print("       - Validation MAE Comparison")
    print("       - Validation RMSE Comparison")
    print("       - Validation R² Comparison")
    print("       - Train vs Val MAE Comparison")
    print("       - Train vs Val RMSE Comparison")
    print("       - Train vs Val R² Comparison")
    print("       - All Metrics Heatmap")
    print("       - Error Boxplot (All Models)")
    print("       - Ensemble Improvement Analysis")
    print("       - Model Ranking by MAE")
    print("       - R² Train vs Val Scatter")
    print("       - Overfitting Analysis")
    print("       - Architecture Comparison")
    print("       - Computation & Efficiency Comparison (Training Time, Inference Time, Efficiency Score)")
    
    print(f"\n🎉 All {total_plots} plots saved in 11 zip files in /kaggle/working/")
    print("="*70)
    
    # Save to CSV
    csv_path = '/kaggle/working/model_comparison_6models_3ensembles.csv'
    display_df.to_csv(csv_path, index=False)
    print(f"\n✓ Results saved to {csv_path}")
    
    # Print best model
    best_model = results_df.iloc[0]
    print(f"\n{'='*70}")
    print(f"🏆 BEST MODEL: {best_model['display_name']}")
    print(f"{'='*70}")
    print(f"Validation MAE: {best_model['val_mae']:.2f} kg")
    print(f"Validation RMSE: {best_model['val_rmse']:.2f} kg")
    print(f"Validation R²: {best_model['val_r2']:.6f}")
    print(f"Trained for {best_model['epochs_trained']} epochs")
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE!")
    print("="*70)

if __name__ == "__main__":
    main()


2026-01-25 10:31:51.032964: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769337111.216424      54 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769337111.267781      54 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769337111.705436      54 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769337111.705483      54 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769337111.705486      54 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

CATTLE WEIGHT PREDICTION - 6 MODELS + 3 ENSEMBLES

LOADING DATASET
Loading metadata from: /kaggle/input/cattle4sides/cattleMetadata.csv
Loading images from: /kaggle/input/cattle4sides/cattle4sides/4 Sides

Dataset Info:
Total records: 215
Columns: ['ID', 'Age (Year)', 'Weight(KG)', 'Sex', 'Breed']
Weight range: 59.00 - 654.00 kg

Loading images...
Processed 50/215 records...
Processed 100/215 records...
Processed 150/215 records...
Processed 200/215 records...

Dataset loaded successfully!
Valid samples: 215
Input shape: (215, 224, 224, 12)
Target shape: (215,)
Weight range: 59.00 - 654.00 kg

Train samples: 172
Validation samples: 43

GENERATING DATASET VISUALIZATION PLOTS

1. Weight Distribution Plots...
2. Weight Comparison Plots...
3. Dataset Statistics...
4. Train/Validation Split Visualization...
5. Sample Images Visualization...

✓ Saved 8 dataset plots to /kaggle/workin

I0000 00:00:1769337217.976972      54 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



After Cleanup:
  RAM: 2196.94 MB (8.8% system usage)
  GPU: Current=0.00 MB, Peak=0.00 MB

RAM Freed: -104.41 MB
GPU Freed: 0.00 MB


Building model: DenseNet121_v1_original
  Config: LR=0.001, Fine-tune=False
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  All base layers frozen

Training DenseNet121_v1_original
Epoch 1/50


I0000 00:00:1769337262.619736     100 service.cc:152] XLA service 0x7c2b05290da0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769337262.619776     100 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1769337275.339486     100 cuda_dnn.cc:529] Loaded cuDNN version 91002


...

11/11 ━━━━━━━━━━━━━━━━━━━━ 721s 31s/step - loss: 81369.8828 - mae: 258.3125 - mse: 81369.8828 - val_loss: 14557.7852 - val_mae: 101.5117 - val_mse: 14557.7852 - learning_rate: 5.0000e-04
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 260ms/step - loss: 22344.5859 - mae: 125.2211 - mse: 22344.5859 - val_loss: 13552.5898 - val_mae: 98.1959 - val_mse: 13552.5898 - learning_rate: 5.0000e-04
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 251ms/step - loss: 12220.4434 - mae: 88.0723 - mse: 12220.4434 - val_loss: 8222.2412 - val_mae: 75.4885 - val_mse: 8222.2412 - learning_rate: 5.0000e-04
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 238ms/step - loss: 5748.2119 - mae: 60.9679 - mse: 5748.2119 - val_loss: 23789.0020 - val_mae: 138.6104 - val_mse: 23789.0020 - learning_rate: 5.0000e-04
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 259ms/step - loss: 8271.1826 - mae: 70.3984 - mse: 8271.1826 - val_loss: 2119.3254 - val_mae: 35.5596 - val_mse: 2119.3254 - learning_rate: 5.0000e-04
Epoch 6/50
11/11 ━━━━━━━━━━━━━━━━━━

2026-01-25 10:59:48.751800: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-25 10:59:48.947341: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


10/11 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - loss: 78522.9297 - mae: 252.9950 - mse: 78522.9297

2026-01-25 11:00:17.913926: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-25 11:00:18.109658: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


11/11 ━━━━━━━━━━━━━━━━━━━━ 106s 5s/step - loss: 76450.1953 - mae: 247.2297 - mse: 76450.1953 - val_loss: 28142.3281 - val_mae: 147.3666 - val_mse: 28142.3281 - learning_rate: 5.0000e-04
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 106ms/step - loss: 11783.8369 - mae: 86.7819 - mse: 11783.8369 - val_loss: 6218.2695 - val_mae: 56.4442 - val_mse: 6218.2695 - learning_rate: 5.0000e-04
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step - loss: 11404.9541 - mae: 82.0638 - mse: 11404.9541 - val_loss: 16794.3828 - val_mae: 105.9529 - val_mse: 16794.3828 - learning_rate: 5.0000e-04
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - loss: 5465.2627 - mae: 58.5229 - mse: 5465.2627 - val_loss: 16673.2617 - val_mae: 105.5878 - val_mse: 16673.2617 - learning_rate: 5.0000e-04
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step - loss: 6885.8696 - mae: 61.7065 - mse: 6885.8696 - val_loss: 17708.3145 - val_mae: 107.8269 - val_mse: 17708.3145 - learning_rate: 5.0000e-04
Epoch 6/50
11/11 ━━━━━━━━━━━━━━━━━━━